In [1]:
# frozen multi-passage retrieval challenge.
from pathlib import Path
from IPython.display import display
import hashlib, json
import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_FOLDER = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "qa_benchmark"
EVALUATION_CORPUS_HASH = "7d74b8baa896be4a5be6a449b0e5eed070a0a67624609630f6c028abce099ded"
CORPUS_TAG = EVALUATION_CORPUS_HASH[:10]

assert DATA_FOLDER.exists(), DATA_FOLDER
assert OUTPUT_ROOT.exists(), OUTPUT_ROOT

In [4]:
# Freeze definitions before viewing method results.
EXPERIMENT_VERSION = "2.4.1"
CHALLENGE_VERSION = "multi-passage-v1"
MULTI_PASSAGE_SCORING_VERSION = "cumulative-required-span-v1"
SENTENCE_WINDOW_PROVENANCE_VERSION = "sentence-window-multispan-v3"
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
OFFSET_CONVENTION = "zero-based-end-exclusive"
PASSAGE_RECOVERY_RULE = "covered_characters_equals_trimmed_gold_characters"
EVALUATION_K_VALUES = (1, 3, 5, 10)
FIXED_CONTEXT_TOKEN_BUDGET = 1000
BUDGET_CANDIDATE_K = 50
FROZEN_METHODS = ("hierarchical", "semantic", "sentence", "sentence_window", "token")
APPROVED_PASSAGE_ROLES = {"core", "definition", "condition", "exception", "implementation"}

print("Project root:", PROJECT_ROOT)
print("Notebook 08 protocol loaded.")
print("Offset convention:", OFFSET_CONVENTION)
print("Recovery rule:", PASSAGE_RECOVERY_RULE)

# Freeze the legal-dependency annotation protocol.
APPROVED_DEPENDENCY_TYPES = {"within_document", "cross_document"}
APPROVED_REVIEW_STATUSES = {"approved", "needs_revision", "rejected"}
REQUIRED_PASSAGE_TEST = "manual_leave_one_passage_out_v1"

print("Required-passage test:", REQUIRED_PASSAGE_TEST)

Project root: /data/home/zll/xh0862/esg_rag_project
Notebook 08 protocol loaded.
Offset convention: zero-based-end-exclusive
Recovery rule: covered_characters_equals_trimmed_gold_characters
Required-passage test: manual_leave_one_passage_out_v1


In [5]:
# Verify the frozen single-passage evaluation without rehashing its large files.
NOTEBOOK07_FOLDER = OUTPUT_ROOT / f"evaluation_single_passage_{CORPUS_TAG}"
NOTEBOOK07_MANIFEST_PATH = NOTEBOOK07_FOLDER / f"evaluation_final_manifest_{CORPUS_TAG}.json"

assert NOTEBOOK07_MANIFEST_PATH.exists(), NOTEBOOK07_MANIFEST_PATH
notebook07_manifest = json.loads(NOTEBOOK07_MANIFEST_PATH.read_text(encoding="utf-8"))

assert notebook07_manifest["corpus_hash"] == EVALUATION_CORPUS_HASH
assert notebook07_manifest["questions"] == 118
assert notebook07_manifest["methods"] == 5
assert notebook07_manifest["retrieval_records"] == 590
assert notebook07_manifest["multi_passage_included"] is False

print("Notebook 07 manifest verified.")
print("Corpus:", notebook07_manifest["corpus_hash"][:10])
print("Questions:", notebook07_manifest["questions"])
print("Notebook 07 indexes will be treated as pre-existing infrastructure.")

Notebook 07 manifest verified.
Corpus: 7d74b8baa8
Questions: 118
Notebook 07 indexes will be treated as pre-existing infrastructure.


In [6]:
# Locate likely challenge files before assuming a schema.
supported_suffixes = {".json", ".jsonl", ".csv", ".parquet", ".xlsx"}
keywords = ("multi", "passage", "challenge", "dependency")

candidate_paths = sorted({
    path for path in DATA_FOLDER.rglob("*")
    if path.is_file()
    and path.suffix.lower() in supported_suffixes
    and any(keyword in path.name.lower() for keyword in keywords)
})

candidate_files_df = pd.DataFrame([
    {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "suffix": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
    }
    for path in candidate_paths
])

if candidate_files_df.empty:
    print("No likely multi-passage input file found.")
else:
    display(candidate_files_df)
    print("Candidate files:", len(candidate_files_df))

No likely multi-passage input file found.


In [8]:
# Inspect representative records before designing the challenge benchmark.
PROVISIONS_PATH = DATA_FOLDER / "parsed/eval_provisions.jsonl"
UNITS_PATH = DATA_FOLDER / "parsed/eval_units.jsonl"

def inspect_jsonl(path, records=2):
    samples = []
    with path.open(encoding="utf-8") as file:
        for _ in range(records):
            line = file.readline()
            if not line: break
            samples.append(json.loads(line))
    print(f"\n{path.name}: {sum(1 for _ in path.open(encoding='utf-8'))} records")
    for i, record in enumerate(samples, 1):
        print(f"Record {i} keys:", list(record))
        display(pd.DataFrame([{
            key: str(value)[:300] + ("..." if len(str(value)) > 300 else "")
            for key, value in record.items()
        }]).T.rename(columns={0: "value"}))

inspect_jsonl(PROVISIONS_PATH)
inspect_jsonl(UNITS_PATH)


eval_provisions.jsonl: 9912 records
Record 1 keys: ['provision_id', 'parent_unit_id', 'doc_id', 'provision_type', 'provision_number', 'provision_title', 'parent_clause_id', 'start_char', 'end_char', 'char_count', 'provision_text', 'official_number', 'cleaned_filename', 'source_docx_filename', 'split', 'document_type', 'esg_domains', 'esg_categories', 'publication_year', 'regulatory_family_id']


,value
provision_id,26_2023_ND-CP_m_570099_article_0002_clause_001
parent_unit_id,26_2023_ND-CP_m_570099_article_0002
doc_id,26_2023_ND-CP_m_570099
provision_type,clause
provision_number,1
provision_title,
parent_clause_id,
start_char,1384
end_char,1440
char_count,56


Record 2 keys: ['provision_id', 'parent_unit_id', 'doc_id', 'provision_type', 'provision_number', 'provision_title', 'parent_clause_id', 'start_char', 'end_char', 'char_count', 'provision_text', 'official_number', 'cleaned_filename', 'source_docx_filename', 'split', 'document_type', 'esg_domains', 'esg_categories', 'publication_year', 'regulatory_family_id']


,value
provision_id,26_2023_ND-CP_m_570099_article_0002_clause_002
parent_unit_id,26_2023_ND-CP_m_570099_article_0002
doc_id,26_2023_ND-CP_m_570099
provision_type,clause
provision_number,2
provision_title,
parent_clause_id,
start_char,1440
end_char,1486
char_count,46



eval_units.jsonl: 1395 records
Record 1 keys: ['unit_id', 'doc_id', 'source_filename', 'unit_type', 'unit_number', 'unit_number_occurrence', 'has_repeated_number', 'unit_title', 'part_number', 'part_title', 'chapter_number', 'chapter_title', 'section_number', 'section_title', 'subsection_number', 'subsection_title', 'start_char', 'end_char', 'char_count', 'is_oversized', 'unit_text', 'official_number', 'cleaned_filename', 'source_docx_filename', 'split', 'document_type', 'esg_domains', 'esg_categories', 'publication_year', 'regulatory_family_id']


,value
unit_id,26_2023_ND-CP_m_570099_article_0001
doc_id,26_2023_ND-CP_m_570099
source_filename,26_2023_ND-CP_m_570099.txt
unit_type,article
unit_number,1
unit_number_occurrence,1
has_repeated_number,False
unit_title,Scope
part_number,
part_title,


Record 2 keys: ['unit_id', 'doc_id', 'source_filename', 'unit_type', 'unit_number', 'unit_number_occurrence', 'has_repeated_number', 'unit_title', 'part_number', 'part_title', 'chapter_number', 'chapter_title', 'section_number', 'section_title', 'subsection_number', 'subsection_title', 'start_char', 'end_char', 'char_count', 'is_oversized', 'unit_text', 'official_number', 'cleaned_filename', 'source_docx_filename', 'split', 'document_type', 'esg_domains', 'esg_categories', 'publication_year', 'regulatory_family_id']


,value
unit_id,26_2023_ND-CP_m_570099_article_0002
doc_id,26_2023_ND-CP_m_570099
source_filename,26_2023_ND-CP_m_570099.txt
unit_type,article
unit_number,2
unit_number_occurrence,1
has_repeated_number,False
unit_title,Regulated entities
part_number,
part_title,


In [9]:
# Load the article-level units and clause-level passages.
units_df = pd.read_json(UNITS_PATH, lines=True)
provisions_df = pd.read_json(PROVISIONS_PATH, lines=True)

assert units_df["unit_id"].is_unique and provisions_df["provision_id"].is_unique
assert units_df["split"].eq("eval").all() and provisions_df["split"].eq("eval").all()
assert (units_df["start_char"] < units_df["end_char"]).all()
assert (provisions_df["start_char"] < provisions_df["end_char"]).all()

# Every provision must belong to its stated parent article and document.
unit_lookup = units_df.set_index("unit_id")[["doc_id", "start_char", "end_char"]]
joined = provisions_df.join(unit_lookup, on="parent_unit_id", rsuffix="_unit")
assert joined["doc_id"].eq(joined["doc_id_unit"]).all()
assert joined["start_char"].ge(joined["start_char_unit"]).all()
assert joined["end_char"].le(joined["end_char_unit"]).all()

print("Units:", len(units_df), "| Provisions:", len(provisions_df))
print("Documents:", units_df["doc_id"].nunique(), "| Regulatory families:", units_df["regulatory_family_id"].nunique())
print("Evidence hierarchy validated.")

Units: 1395 | Provisions: 9912
Documents: 60 | Regulatory families: 47
Evidence hierarchy validated.


In [10]:
# Summarise documents and regulatory families that can support multi-passage questions.
document_candidates_df = units_df.groupby(["doc_id", "official_number", "regulatory_family_id"], as_index=False).agg(
    articles=("unit_id", "nunique"), provisions=("unit_id", lambda ids: provisions_df["parent_unit_id"].isin(ids).sum()),
    categories=("esg_categories", "first")
).query("articles >= 2").sort_values(["articles", "provisions"], ascending=False)

family_candidates_df = units_df.groupby("regulatory_family_id", as_index=False).agg(
    documents=("doc_id", "nunique"), articles=("unit_id", "nunique"),
    official_numbers=("official_number", lambda values: sorted(set(map(str, values))))
).query("documents >= 2").sort_values(["documents", "articles"], ascending=False)

print("Within-document candidates:", len(document_candidates_df))
display(document_candidates_df.head(20))
print("Cross-document regulatory families:", len(family_candidates_df))
display(family_candidates_df.head(20))

Within-document candidates: 60


,doc_id,official_number,regulatory_family_id,articles,provisions,categories
14,145_2020_ND-CP_m_461788,145/2020/ND-CP,family_0086,115,764,"[Board, Collective Bargaining, Public Health, ..."
53,68_VBHN-VPQH_m_716092,68/VBHN-VPQH,family_0249,107,767,"[Biodiversity, Diversity, Forests, Strategy, U..."
27,23_2004_QH11_m_76135,23/2004/QH11,family_0137,103,519,"[Board, Waste, Water]"
40,36_2024_QH15_620124,36/2024/QH15,family_0188,89,883,"[Board, Corruption, Energy, Waste, Water]"
50,64_2025_QH15_m_646832,64/2025/QH15,family_0240,72,504,"[Corruption, Human Rights, Taxes, Waste]"
3,05_2007_QH12_m_85253,05/2007/QH12,family_0029,72,389,[Energy]
52,68_2006_QH11_m_80643,68/2006/QH11,family_0248,71,327,"[Board, Unions, Waste]"
4,07_2017_QH14_m_355880,07/2017/QH14,family_0035,60,450,"[Biodiversity, Board, Diversity, Energy, Hazar..."
48,50_2010_QH12_m_113380,50/2010/QH12,family_0218,48,230,"[Energy, Packaging, Waste, Water]"
45,40_2025_TT-BNNMT_m_673353,40/2025/TT-BNNMT,family_0028,44,297,"[Waste, Water]"


Cross-document regulatory families: 8


,regulatory_family_id,documents,articles,official_numbers
9,family_0058,5,21,"[108/2025/ND-CP, 144/2024/ND-CP, 199/2025/ND-C..."
2,family_0028,4,82,"[04/2026/TT-BNNMT, 36/2025/TT-BNNMT, 39/2025/T..."
17,family_0101,2,76,"[165/2025/ND-CP, 356/2025/ND-CP]"
3,family_0029,2,75,"[05/2007/QH12, 78/2025/QH15]"
42,family_0248,2,74,"[68/2006/QH11, 70/2025/QH15]"
15,family_0093,2,32,"[152/2020/ND-CP, 70/2023/ND-CP]"
20,family_0114,2,21,"[17/2026/TT-BNNMT, 25/2018/TT-BNNPTNT]"
5,family_0040,2,19,"[08/2020/QD-TTg, 24/2014/QD-TTg]"


In [12]:
# Display related articles for manual legal-dependency review.

import re
def review_units(scope, identifier, terms=None, limit=30):
    column = "doc_id" if scope == "within_document" else "regulatory_family_id"
    review = units_df.loc[units_df[column].astype(str).eq(str(identifier))].copy()
    if terms:
        pattern = "|".join(map(re.escape, terms))
        review = review[review["unit_text"].str.contains(pattern, case=False, na=False, regex=True)]
    review["text_preview"] = review["unit_text"].str.replace("\n", " ", regex=False).str[:600]
    columns = ["unit_id", "doc_id", "official_number", "unit_number", "unit_title", "text_preview"]
    print(scope, identifier, "| matching articles:", len(review))
    display(review[columns].head(limit))

# Initial manageable candidates; identifiers can be changed after inspection.
review_units("within_document", "17_2022_TT-BTNMT_m_550994",
             terms=["except", "unless", "condition", "responsibility", "shall", "must"])
review_units("cross_document", "family_0114")

within_document 17_2022_TT-BTNMT_m_550994 | matching articles: 26


,unit_id,doc_id,official_number,unit_number,unit_title,text_preview
1355,17_2022_TT-BTNMT_m_550994_article_0005,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,5,Identification of method for sector-level GHG ...,Article 5. Identification of method for sector...
1356,17_2022_TT-BTNMT_m_550994_article_0006,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,6,Selection of sector-level GHG emission factors,Article 6. Selection of sector-level GHG emiss...
1358,17_2022_TT-BTNMT_m_550994_article_0008,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,8,Calculation of sector-level GHG emission and a...,Article 8. Calculation of sector-level GHG emi...
1360,17_2022_TT-BTNMT_m_550994_article_0010,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,10,Assurance of quality of sector-level GHG inven...,Article 10. Assurance of quality of sector-lev...
1361,17_2022_TT-BTNMT_m_550994_article_0011,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,11,Assessment of uncertainties of sector-level GH...,Article 11. Assessment of uncertainties of sec...
1362,17_2022_TT-BTNMT_m_550994_article_0012,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,12,Recalculation of sector-level GHG inventories,Article 12. Recalculation of sector-level GHG ...
1363,17_2022_TT-BTNMT_m_550994_article_0013,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,13,Preparation of reports on sector-level GHG inv...,Article 13. Preparation of reports on sector-l...
1365,17_2022_TT-BTNMT_m_550994_article_0015,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,15,Identification of operational boundaries and s...,Article 15. Identification of operational boun...
1367,17_2022_TT-BTNMT_m_550994_article_0017,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,17,Selection of facility-level GHG emission factors,Article 17. Selection of facility-level GHG em...
1368,17_2022_TT-BTNMT_m_550994_article_0018,17_2022_TT-BTNMT_m_550994,17/2022/TT-BTNMT,18,Selection and collection of activity data for ...,Article 18. Selection and collection of activi...


cross_document family_0114 | matching articles: 21


,unit_id,doc_id,official_number,unit_number,unit_title,text_preview
103,25_2018_TT-BNNPTNT_m_411295_article_0001,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,1,Scope,Article 1. Scope This Circular provides guidel...
104,25_2018_TT-BNNPTNT_m_411295_article_0002,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,2,Regulated entities,Article 2. Regulated entities This Circular ap...
105,25_2018_TT-BNNPTNT_m_411295_article_0003,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,3,Definitions,Article 3. Definitions For the purposes of thi...
106,25_2018_TT-BNNPTNT_m_411295_article_0004,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,4,Cases of license for import of live aquatic an...,Article 4. Cases of license for import of live...
107,25_2018_TT-BNNPTNT_m_411295_article_0005,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,5,Procedures for licensing import of live aquati...,Article 5. Procedures for licensing import of ...
108,25_2018_TT-BNNPTNT_m_411295_article_0006,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,6,Procedures for licensing import of live aquati...,Article 6. Procedures for licensing import of ...
109,25_2018_TT-BNNPTNT_m_411295_article_0007,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,7,Effective period of license for import of live...,Article 7. Effective period of license for imp...
110,25_2018_TT-BNNPTNT_m_411295_article_0008,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,8,Contents of risk assessment of imported live a...,Article 8. Contents of risk assessment of impo...
111,25_2018_TT-BNNPTNT_m_411295_article_0009,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,9,Risk assessment methods,Article 9. Risk assessment methods 1. Identifi...
112,25_2018_TT-BNNPTNT_m_411295_article_0010,25_2018_TT-BNNPTNT_m_411295,25/2018/TT-BNNPTNT,10,Establishment of risk assessment council,Article 10. Establishment of risk assessment c...


In [13]:
# Prioritise clauses likely to qualify or modify another legal rule.
review_units("within_document", "17_2022_TT-BTNMT_m_550994",
             terms=["except", "unless", "subject to", "provided that", "condition"],
             limit=15)

within_document 17_2022_TT-BTNMT_m_550994 | matching articles: 0


,unit_id,doc_id,official_number,unit_number,unit_title,text_preview


In [15]:
# Find documents containing both definition-like and obligation-like articles.
definition_pattern = r"\bmeans\b|\brefers to\b|\bis defined as\b|\bis understood as\b|\bfor the purposes of\b"
obligation_pattern = r"\bshall\b|\bmust\b|\bis required to\b|\bare required to\b|\bresponsible for\b"

candidate_units_df = units_df.copy()
candidate_units_df["definition_hit"] = candidate_units_df["unit_text"].str.contains(definition_pattern, case=False, na=False, regex=True)
candidate_units_df["obligation_hit"] = candidate_units_df["unit_text"].str.contains(obligation_pattern, case=False, na=False, regex=True)

within_relationships_df = candidate_units_df.groupby(
    ["doc_id", "official_number", "regulatory_family_id"], as_index=False
).agg(
    articles=("unit_id", "nunique"),
    definition_articles=("definition_hit", "sum"),
    obligation_articles=("obligation_hit", "sum"),
    categories=("esg_categories", "first")
)

within_relationships_df = within_relationships_df.query(
    "definition_articles > 0 and obligation_articles > 0"
).sort_values(["definition_articles", "obligation_articles"], ascending=False)

print("Documents with definition + obligation candidates:", len(within_relationships_df))
display(within_relationships_df.head(30))

Documents with definition + obligation candidates: 31


,doc_id,official_number,regulatory_family_id,articles,definition_articles,obligation_articles,categories
59,85_2025_TT-BNNMT_m_694911,85/2025/TT-BNNMT,family_0258,39,10,35,"[Biodiversity, Board, Diversity, Forests, Wast..."
27,23_2004_QH11_m_76135,23/2004/QH11,family_0137,103,8,87,"[Board, Waste, Water]"
40,36_2024_QH15_620124,36/2024/QH15,family_0188,89,7,79,"[Board, Corruption, Energy, Waste, Water]"
14,145_2020_ND-CP_m_461788,145/2020/ND-CP,family_0086,115,4,86,"[Board, Collective Bargaining, Public Health, ..."
46,43_2025_TT-BCT_m_674734,43/2025/TT-BCT,family_0207,11,4,7,[Water]
53,68_VBHN-VPQH_m_716092,68/VBHN-VPQH,family_0249,107,3,98,"[Biodiversity, Diversity, Forests, Strategy, U..."
50,64_2025_QH15_m_646832,64/2025/QH15,family_0240,72,3,67,"[Corruption, Human Rights, Taxes, Waste]"
4,07_2017_QH14_m_355880,07/2017/QH14,family_0035,60,3,48,"[Biodiversity, Board, Diversity, Energy, Hazar..."
38,356_2025_ND-CP_689146,356/2025/ND-CP,family_0101,42,3,31,"[Climate Risk Management, Systemic Risk, Taxes]"
34,26_2025_TT-BNNMT_m_673079,26/2025/TT-BNNMT,family_0155,33,3,28,"[Biodiversity, Board, Diversity, Forests, Publ..."


In [16]:
# Find definition and obligation evidence in different documents from the same regulatory family.
definition_docs = candidate_units_df[candidate_units_df["definition_hit"]].groupby(
    ["regulatory_family_id", "doc_id", "official_number"], as_index=False
).agg(definition_articles=("unit_id", "nunique"))

obligation_docs = candidate_units_df[candidate_units_df["obligation_hit"]].groupby(
    ["regulatory_family_id", "doc_id", "official_number"], as_index=False
).agg(obligation_articles=("unit_id", "nunique"))

cross_relationships_df = definition_docs.merge(
    obligation_docs, on="regulatory_family_id", suffixes=("_definition", "_obligation")
).query("doc_id_definition != doc_id_obligation").sort_values(
    ["definition_articles", "obligation_articles"], ascending=False
)

print("Cross-document candidate relationships:", len(cross_relationships_df))
display(cross_relationships_df.head(30))

Cross-document candidate relationships: 16


,regulatory_family_id,doc_id_definition,official_number_definition,definition_articles,doc_id_obligation,official_number_obligation,obligation_articles
24,family_0101,165_2025_ND-CP_m_664591,165/2025/ND-CP,3,356_2025_ND-CP_689146,356/2025/ND-CP,31
25,family_0101,356_2025_ND-CP_689146,356/2025/ND-CP,3,165_2025_ND-CP_m_664591,165/2025/ND-CP,24
42,family_0248,68_2006_QH11_m_80643,68/2006/QH11,2,70_2025_QH15_m_674897,70/2025/QH15,3
10,family_0029,05_2007_QH12_m_85253,05/2007/QH12,2,78_2025_QH15_683248,78/2025/QH15,2
19,family_0093,152_2020_ND-CP_m_461585,152/2020/ND-CP,2,70_2023_ND-CP_m_580154,70/2023/ND-CP,1
43,family_0248,70_2025_QH15_m_674897,70/2025/QH15,1,68_2006_QH11_m_80643,68/2006/QH11,45
11,family_0029,78_2025_QH15_683248,78/2025/QH15,1,05_2007_QH12_m_85253,05/2007/QH12,43
4,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,1,40_2025_TT-BNNMT_m_673353,40/2025/TT-BNNMT,40
6,family_0028,40_2025_TT-BNNMT_m_673353,40/2025/TT-BNNMT,1,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,22
20,family_0093,70_2023_ND-CP_m_580154,70/2023/ND-CP,1,152_2020_ND-CP_m_461585,152/2020/ND-CP,21


In [17]:
# Review definition/obligation articles in the strongest cross-document family.
family_id = "family_0101"
family_review_df = candidate_units_df[
    candidate_units_df["regulatory_family_id"].eq(family_id)
    & (candidate_units_df["definition_hit"] | candidate_units_df["obligation_hit"])
].copy()

family_review_df["candidate_role"] = family_review_df.apply(
    lambda row: "definition + obligation" if row["definition_hit"] and row["obligation_hit"]
    else "definition" if row["definition_hit"] else "obligation", axis=1
)
family_review_df["text_preview"] = family_review_df["unit_text"].str.replace("\n", " ", regex=False).str[:900]

display(family_review_df[
    ["candidate_role", "unit_id", "doc_id", "official_number", "unit_number", "unit_title", "text_preview"]
].sort_values(["official_number", "candidate_role"]))

,candidate_role,unit_id,doc_id,official_number,unit_number,unit_title,text_preview
245,definition + obligation,165_2025_ND-CP_m_664591_article_0006,165_2025_ND-CP_m_664591,165/2025/ND-CP,6,Data access and retrieval,Article 6. Data access and retrieval 1. Data a...
249,definition + obligation,165_2025_ND-CP_m_664591_article_0010,165_2025_ND-CP_m_664591,165/2025/ND-CP,10,Data disclosure,Article 10. Data disclosure 1. The disclosure ...
252,definition + obligation,165_2025_ND-CP_m_664591_article_0013,165_2025_ND-CP_m_664591,165/2025/ND-CP,13,Other operations in data processing,Article 13. Other operations in data processin...
242,obligation,165_2025_ND-CP_m_664591_article_0003,165_2025_ND-CP_m_664591,165/2025/ND-CP,3,Criteria for determining important data,Article 3. Criteria for determining important ...
243,obligation,165_2025_ND-CP_m_664591_article_0004,165_2025_ND-CP_m_664591,165/2025/ND-CP,4,Criteria for determining core data,Article 4. Criteria for determining core data ...
244,obligation,165_2025_ND-CP_m_664591_article_0005,165_2025_ND-CP_m_664591,165/2025/ND-CP,5,Data storage,Article 5. Data storage 1. Data owners shall s...
246,obligation,165_2025_ND-CP_m_664591_article_0007,165_2025_ND-CP_m_664591,165/2025/ND-CP,7,Support for data owners in data connection and...,Article 7. Support for data owners in data con...
247,obligation,165_2025_ND-CP_m_664591_article_0008,165_2025_ND-CP_m_664591,165/2025/ND-CP,8,Provision of data for state agencies,Article 8. Provision of data for state agencie...
248,obligation,165_2025_ND-CP_m_664591_article_0009,165_2025_ND-CP_m_664591,165/2025/ND-CP,9,Data confirmation and authentication,Article 9. Data confirmation and authenticatio...
250,obligation,165_2025_ND-CP_m_664591_article_0011,165_2025_ND-CP_m_664591,165/2025/ND-CP,11,Data encryption and decryption,Article 11. Data encryption and decryption 1. ...


In [18]:
# Export complete cross-document article pairs for review.
definition_units = candidate_units_df[candidate_units_df["definition_hit"]][
    ["regulatory_family_id", "doc_id", "official_number", "unit_id", "unit_number", "unit_title", "unit_text"]
].rename(columns={c: f"definition_{c}" for c in ["doc_id", "official_number", "unit_id", "unit_number", "unit_title", "unit_text"]})

obligation_units = candidate_units_df[candidate_units_df["obligation_hit"]][
    ["regulatory_family_id", "doc_id", "official_number", "unit_id", "unit_number", "unit_title", "unit_text"]
].rename(columns={c: f"obligation_{c}" for c in ["doc_id", "official_number", "unit_id", "unit_number", "unit_title", "unit_text"]})

cross_pair_review_df = definition_units.merge(obligation_units, on="regulatory_family_id")
cross_pair_review_df = cross_pair_review_df.query("definition_doc_id != obligation_doc_id").drop_duplicates(
    ["definition_unit_id", "obligation_unit_id"]
).sort_values(["regulatory_family_id", "definition_official_number", "obligation_official_number"]).reset_index(drop=True)

cross_pair_review_df.insert(0, "candidate_pair_id", [f"CROSS-{i:03d}" for i in range(1, len(cross_pair_review_df) + 1)])
CHALLENGE_FOLDER = OUTPUT_ROOT / f"multi_passage_challenge_{CORPUS_TAG}"
CHALLENGE_FOLDER.mkdir(parents=True, exist_ok=True)
CROSS_REVIEW_PATH = CHALLENGE_FOLDER / f"cross_document_candidate_review_{CORPUS_TAG}.csv"
cross_pair_review_df.to_csv(CROSS_REVIEW_PATH, index=False, encoding="utf-8-sig")

print("Candidate pairs:", len(cross_pair_review_df))
print("Saved:", CROSS_REVIEW_PATH)
display(cross_pair_review_df.head(10))

Candidate pairs: 375
Saved: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/multi_passage_challenge_7d74b8baa8/cross_document_candidate_review_7d74b8baa8.csv


,candidate_pair_id,regulatory_family_id,definition_doc_id,definition_official_number,definition_unit_id,definition_unit_number,definition_unit_title,definition_unit_text,obligation_doc_id,obligation_official_number,obligation_unit_id,obligation_unit_number,obligation_unit_title,obligation_unit_text
0,CROSS-001,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,04_2026_TT-BNNMT_m_703639,04/2026/TT-BNNMT,04_2026_TT-BNNMT_m_703639_article_0002,2,Amendments and supplements to Article 6,Article 2. Amendments and supplements to Artic...
1,CROSS-002,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,04_2026_TT-BNNMT_m_703639,04/2026/TT-BNNMT,04_2026_TT-BNNMT_m_703639_article_0004,4,Amendments and supplements to Article 18,Article 4. Amendments and supplements to Artic...
2,CROSS-003,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,04_2026_TT-BNNMT_m_703639,04/2026/TT-BNNMT,04_2026_TT-BNNMT_m_703639_article_0005,5,Amendments and supplements to Article 20,Article 5. Amendments and supplements to Artic...
3,CROSS-004,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,39_2025_TT-BNNMT_m_666465,39/2025/TT-BNNMT,39_2025_TT-BNNMT_m_666465_article_0003,3,Contents of mine closure schemes,Article 3. Contents of mine closure schemes\n1...
4,CROSS-005,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,39_2025_TT-BNNMT_m_666465,39/2025/TT-BNNMT,39_2025_TT-BNNMT_m_666465_article_0004,4,Contents of mine closure plans,Article 4. Contents of mine closure plans\n1. ...
5,CROSS-006,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,39_2025_TT-BNNMT_m_666465,39/2025/TT-BNNMT,39_2025_TT-BNNMT_m_666465_article_0005,5,Templates of documents included in application...,Article 5. Templates of documents included in ...
6,CROSS-007,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,39_2025_TT-BNNMT_m_666465,39/2025/TT-BNNMT,39_2025_TT-BNNMT_m_666465_article_0007,7,Effect,Article 7. Effect\n1. This Circular comes into...
7,CROSS-008,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,39_2025_TT-BNNMT_m_666465,39/2025/TT-BNNMT,39_2025_TT-BNNMT_m_666465_article_0008,8,Responsibility for implementation,Article 8. Responsibility for implementation\n...
8,CROSS-009,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,40_2025_TT-BNNMT_m_673353,40/2025/TT-BNNMT,40_2025_TT-BNNMT_m_673353_article_0003,3,Interpretation of terms,Article 3. Interpretation of terms\nFor the pu...
9,CROSS-010,family_0028,36_2025_TT-BNNMT_m_679483,36/2025/TT-BNNMT,36_2025_TT-BNNMT_m_679483_article_0003,3,Term interpretation,Article 3. Term interpretation\nIn this Circul...,40_2025_TT-BNNMT_m_673353,40/2025/TT-BNNMT,40_2025_TT-BNNMT_m_673353_article_0004,4,Requirements for the calculation of reserves a...,Article 4. Requirements for the calculation of...


In [19]:
# Preserve only plausible cross-document candidates for clause-level review.
review_status = {
    "CROSS-224": ("strong", "definition+obligation", "Deletion/destruction definitions clarify the required big-data policy."),
    "CROSS-323": ("strong", "definition+implementation", "The base definition applies to the amended procedure."),
    "CROSS-357": ("strong", "definition+condition", "Amended conformity definitions support an apparently unchanged requirement."),
    "CROSS-366": ("strong", "definition+obligation", "Amended accreditation definitions support apparently unchanged obligations."),
    "CROSS-089": ("needs_legal_check", "definition+obligation", "Confirm that Article 20 remains applicable after the 2025 amendment."),
    "CROSS-139": ("needs_legal_check", "condition+implementation", "Confirm the unchanged certification procedure applies with the amended exemption rules."),
    "CROSS-259": ("needs_legal_check", "combined_conditions", "Confirm both data-transfer regimes apply simultaneously."),
    "CROSS-286": ("needs_legal_check", "combined_obligations", "Confirm both protection regimes apply simultaneously.")
}

cross_shortlist_df = cross_pair_review_df[cross_pair_review_df["candidate_pair_id"].isin(review_status)].copy()
cross_shortlist_df[["review_status", "relationship_type", "review_note"]] = cross_shortlist_df["candidate_pair_id"].map(review_status).apply(pd.Series)
cross_shortlist_df = cross_shortlist_df.sort_values(["review_status", "candidate_pair_id"])
CROSS_SHORTLIST_PATH = CHALLENGE_FOLDER / f"cross_document_shortlist_{CORPUS_TAG}.csv"
cross_shortlist_df.to_csv(CROSS_SHORTLIST_PATH, index=False, encoding="utf-8-sig")

print("Shortlisted cross-document pairs:", len(cross_shortlist_df))
print("Saved:", CROSS_SHORTLIST_PATH)
display(cross_shortlist_df[[
    "candidate_pair_id", "review_status", "relationship_type",
    "definition_official_number", "definition_unit_title",
    "obligation_official_number", "obligation_unit_title", "review_note"
]])

Shortlisted cross-document pairs: 8
Saved: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/multi_passage_challenge_7d74b8baa8/cross_document_shortlist_7d74b8baa8.csv


,candidate_pair_id,review_status,relationship_type,definition_official_number,definition_unit_title,obligation_official_number,obligation_unit_title,review_note
88,CROSS-089,needs_legal_check,definition+obligation,78/2025/QH15,Amendments to the Law on product and goods qua...,05/2007/QH12,Obligations of conformity evaluation organizat...,Confirm that Article 20 remains applicable aft...
138,CROSS-139,needs_legal_check,condition+implementation,70/2023/ND-CP,Amendments to the Government’s Decree No. 152/...,152/2020/ND-CP,Certification of exemption from work permit,Confirm the unchanged certification procedure ...
258,CROSS-259,needs_legal_check,combined_conditions,356/2025/ND-CP,Personal data transfer,165/2025/ND-CP,Cross-border data transfer and processing,Confirm both data-transfer regimes apply simul...
285,CROSS-286,needs_legal_check,combined_obligations,356/2025/ND-CP,Personal data protection in big data processing,165/2025/ND-CP,Data protection,Confirm both protection regimes apply simultan...
223,CROSS-224,strong,definition+obligation,165/2025/ND-CP,Other operations in data processing,356/2025/ND-CP,Personal data protection in big data processing,Deletion/destruction definitions clarify the r...
322,CROSS-323,strong,definition+implementation,25/2018/TT-BNNPTNT,Definitions,17/2026/TT-BNNMT,Amendments to some points and clauses of Artic...,The base definition applies to the amended pro...
356,CROSS-357,strong,definition+condition,70/2025/QH15,Amendments to a number of articles of the Law ...,68/2006/QH11,- Requirements for standards and technical reg...,Amended conformity definitions support an appa...
365,CROSS-366,strong,definition+obligation,70/2025/QH15,Amendments to a number of articles of the Law ...,68/2006/QH11,- Rights and obligations of accredited organiz...,Amended accreditation definitions support appa...


In [20]:
# Record candidate decisions before selecting exact gold passages.
candidate_decisions = {
    "CROSS-139": ("retain", "potential_strong", "Base procedure combined with its amendment."),
    "CROSS-323": ("retain", "potential_strong", "Base definition supports the amended procedure."),
    "CROSS-366": ("retain", "potential_strong", "Current obligation may require the original provision and its amendment."),
    "CROSS-259": ("hold", "legal_scope_check", "Both regimes apply only to data that is personal and core/important."),
    "CROSS-286": ("hold", "legal_scope_check", "Requires simultaneous applicability of both protection regimes."),
    "CROSS-224": ("hold", "definition_scope_check", "Definition may not automatically apply across decrees."),
    "CROSS-089": ("reject", "superseded", "Article 20 was repealed by the 2025 amendment."),
    "CROSS-357": ("reject", "superseded", "Article 42 was repealed by the 2025 amendment.")
}

screened_pairs_df = cross_pair_review_df[cross_pair_review_df["candidate_pair_id"].isin(candidate_decisions)].copy()
screened_pairs_df[["decision", "review_class", "decision_reason"]] = screened_pairs_df["candidate_pair_id"].map(candidate_decisions).apply(pd.Series)
screened_pairs_df = screened_pairs_df.sort_values(["decision", "candidate_pair_id"])
SCREENED_PAIRS_PATH = CHALLENGE_FOLDER / f"cross_document_screened_{CORPUS_TAG}.csv"
screened_pairs_df.to_csv(SCREENED_PAIRS_PATH, index=False, encoding="utf-8-sig")

display(screened_pairs_df[["candidate_pair_id", "decision", "review_class", "decision_reason"]])
print("Saved:", SCREENED_PAIRS_PATH)

,candidate_pair_id,decision,review_class,decision_reason
223,CROSS-224,hold,definition_scope_check,Definition may not automatically apply across ...
258,CROSS-259,hold,legal_scope_check,Both regimes apply only to data that is person...
285,CROSS-286,hold,legal_scope_check,Requires simultaneous applicability of both pr...
88,CROSS-089,reject,superseded,Article 20 was repealed by the 2025 amendment.
356,CROSS-357,reject,superseded,Article 42 was repealed by the 2025 amendment.
138,CROSS-139,retain,potential_strong,Base procedure combined with its amendment.
322,CROSS-323,retain,potential_strong,Base definition supports the amended procedure.
365,CROSS-366,retain,potential_strong,Current obligation may require the original pr...


Saved: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/multi_passage_challenge_7d74b8baa8/cross_document_screened_7d74b8baa8.csv


In [21]:
# Expand retained article-level candidates into clause-level passages.
strong_ids = ["CROSS-139", "CROSS-323", "CROSS-366"]
selected_pairs = screened_pairs_df.set_index("candidate_pair_id").loc[strong_ids]
passage_rows = []

for candidate_id, pair in selected_pairs.iterrows():
    for side in ["definition", "obligation"]:
        passages = provisions_df[provisions_df["parent_unit_id"].eq(pair[f"{side}_unit_id"])].copy()
        passages.insert(0, "candidate_side", side)
        passages.insert(0, "candidate_pair_id", candidate_id)
        passage_rows.append(passages)

passage_review_df = pd.concat(passage_rows, ignore_index=True)
passage_review_df["text_preview"] = passage_review_df["provision_text"].str.replace("\n", " ", regex=False).str[:500]
PASSAGE_REVIEW_PATH = CHALLENGE_FOLDER / f"strong_candidate_clause_review_{CORPUS_TAG}.csv"
passage_review_df.to_csv(PASSAGE_REVIEW_PATH, index=False, encoding="utf-8-sig")

print(passage_review_df.groupby(["candidate_pair_id", "candidate_side"]).size())
print("Saved:", PASSAGE_REVIEW_PATH)
display(passage_review_df[[
    "candidate_pair_id", "candidate_side", "provision_id", "provision_number",
    "start_char", "end_char", "char_count", "text_preview"
]])

candidate_pair_id  candidate_side
CROSS-139          definition         65
                   obligation         10
CROSS-323          definition          2
                   obligation          2
CROSS-366          definition        339
                   obligation         10
dtype: int64
Saved: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/multi_passage_challenge_7d74b8baa8/strong_candidate_clause_review_7d74b8baa8.csv


,candidate_pair_id,candidate_side,provision_id,provision_number,start_char,end_char,char_count,text_preview
0,CROSS-139,definition,70_2023_ND-CP_m_580154_article_0001_clause_001,1,1333,2183,850,1. Amendments to certain points and clauses of...
1,CROSS-139,definition,70_2023_ND-CP_m_580154_article_0001_point_001,a,1391,1610,219,a) Amendments to point a clause 3 of Article 3...
2,CROSS-139,definition,70_2023_ND-CP_m_580154_article_0001_point_002,b,1610,1701,91,b) Amendments to clause 5 of Article 3: “5. “e...
3,CROSS-139,definition,70_2023_ND-CP_m_580154_article_0001_point_003,a,1701,1792,91,"a) The head of a branch, representative office..."
4,CROSS-139,definition,70_2023_ND-CP_m_580154_article_0001_point_004,b,1792,1970,178,b) The head who directly administers at least ...
...,...,...,...,...,...,...,...,...
423,CROSS-366,obligation,68_2006_QH11_m_80643_article_0056_point_004,a,50626,50788,162,a/ To ensure conformity of their accredited or...
424,CROSS-366,obligation,68_2006_QH11_m_80643_article_0056_point_005,b,50788,50904,116,b/ To maintain a management system meeting req...
425,CROSS-366,obligation,68_2006_QH11_m_80643_article_0056_point_006,c,50904,50979,75,c/ To ensure objectivity and fairness in confo...
426,CROSS-366,obligation,68_2006_QH11_m_80643_article_0056_point_007,d,50979,51156,177,d/ Conformity certification organizations spec...
